# Telecom Customer Churn & Retention
## 01 — Data Preparation

### Project objective
A telecommunications company wants to understand customer churn, identify the factors associated with customers leaving, predict which current customers are at risk of churn, and prioritize high-value customers for retention efforts.

This notebook prepares the IBM Telco Customer Churn dataset for the later analysis and machine-learning stages.

### Data preparation goals
- Load and inspect the raw customer-level dataset.
- Validate the dataset structure and target variable.
- Check duplicates, missing values, and data types.
- Identify constant, redundant, identifier, high-cardinality, and leakage-prone columns.
- Clean the data without removing fields that are useful for descriptive business analysis.
- Create a separate, leakage-safe dataset for predictive modeling.
- Export cleaned datasets for the next project stages.


In [3]:
# Imports
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 1. Load the dataset


In [4]:
# Locate the raw Excel file
candidate_paths = [Path("../data/raw/Telco_customer_churn.xlsx")]

DATA_PATH = next((path for path in candidate_paths if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Telco_customer_churn.xlsx was not found. "
        "Place it in data/raw/ or in the same folder as this notebook."
    )

df_raw = pd.read_excel(DATA_PATH, sheet_name="Telco_Churn")

print(f"Loaded: {DATA_PATH}")
print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]}")


Loaded: ../data/raw/Telco_customer_churn.xlsx
Rows: 7,043
Columns: 33


In [5]:
# Preview the raw data
df_raw.head()


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.96,-118.27,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.06,-118.31,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.05,-118.29,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.06,-118.32,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,"3,046.05",Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.04,-118.27,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,"5,036.30",Yes,1,89,5340,Competitor had better devices


## 2. Dataset structure

Each row represents one telecom customer. The dataset combines customer demographics, service subscriptions, account information, billing data, churn outcomes, and customer value information.


In [6]:
# Column names and data types
structure = pd.DataFrame({
    "column": df_raw.columns,
    "dtype": df_raw.dtypes.astype(str).values,
    "non_null": df_raw.notna().sum().values,
    "unique_values": df_raw.nunique(dropna=False).values,
})

structure


,column,dtype,non_null,unique_values
0,CustomerID,object,7043,7043
1,Count,int64,7043,1
2,Country,object,7043,1
3,State,object,7043,1
4,City,object,7043,1129
5,Zip Code,int64,7043,1652
6,Lat Long,object,7043,1652
7,Latitude,float64,7043,1652
8,Longitude,float64,7043,1651
9,Gender,object,7043,2


In [7]:
# Numeric summary
df_raw.describe(include=[np.number]).T


,count,mean,std,min,25%,50%,75%,max
Count,"7,043.00",1.00,0.00,1.00,1.00,1.00,1.00,1.00
Zip Code,"7,043.00","93,521.96","1,865.79","90,001.00","92,102.00","93,552.00","95,351.00","96,161.00"
Latitude,"7,043.00",36.28,2.46,32.56,34.03,36.39,38.22,41.96
Longitude,"7,043.00",-119.80,2.16,-124.30,-121.82,-119.73,-118.04,-114.19
Tenure Months,"7,043.00",32.37,24.56,0.00,9.00,29.00,55.00,72.00
Monthly Charges,"7,043.00",64.76,30.09,18.25,35.50,70.35,89.85,118.75
Churn Value,"7,043.00",0.27,0.44,0.00,0.00,0.00,1.00,1.00
Churn Score,"7,043.00",58.70,21.53,5.00,40.00,61.00,75.00,100.00
CLTV,"7,043.00","4,400.30","1,183.06","2,003.00","3,469.00","4,527.00","5,380.50","6,500.00"


In [8]:
# Categorical summary
df_raw.describe(include=["object"]).T


,count,unique,top,freq
CustomerID,7043,7043,3186-AJIEK,1
Country,7043,1,United States,7043
State,7043,1,California,7043
City,7043,1129,Los Angeles,305
Lat Long,7043,1652,"32.67102, -117.095235",5
Gender,7043,2,Male,3555
Senior Citizen,7043,2,No,5901
Partner,7043,2,No,3641
Dependents,7043,2,No,5416
Phone Service,7043,2,Yes,6361


## 3. Data quality checks

Before analysis, check for duplicate customers, missing values, and inconsistent target values.


In [9]:
# Duplicate checks
duplicate_rows = df_raw.duplicated().sum()
duplicate_customer_ids = df_raw["CustomerID"].duplicated().sum()

print(f"Duplicate rows: {duplicate_rows:,}")
print(f"Duplicate CustomerIDs: {duplicate_customer_ids:,}")


Duplicate rows: 0
Duplicate CustomerIDs: 0


In [10]:
# Missing-value report
missing_report = (
    df_raw.isna()
    .sum()
    .to_frame("missing_count")
    .assign(missing_pct=lambda x: 100 * x["missing_count"] / len(df_raw))
    .query("missing_count > 0")
    .sort_values("missing_count", ascending=False)
)

if missing_report.empty:
    print("No native missing values detected.")
else:
    display(missing_report)


,missing_count,missing_pct
Churn Reason,5174,73.46


In [11]:
# Target consistency
target_check = pd.crosstab(
    df_raw["Churn Label"],
    df_raw["Churn Value"],
    margins=True
)

target_check


Churn Value,0,1,All
Churn Label,,,
No,5174,0,5174
Yes,0,1869,1869
All,5174,1869,7043


## 4. Column roles and leakage assessment

**Target**
- `Churn Value` is the binary target: `1 = churned`, `0 = retained`.

**Post-outcome / leakage fields**
- `Churn Label` is another representation of the target.
- `Churn Score` is an existing churn score already derived from churn-related information.
- `Churn Reason` describes why a customer churned and is only known for customers who have already left.

**Non-predictive / redundant fields**
- `CustomerID` is an identifier.
- `Count` is a row-count helper.
- `Country` and `State` are constant in this dataset.
- `Lat Long` duplicates the information contained in `Latitude` and `Longitude`.
- Exact geographic fields such as `City`, `Zip Code`, `Latitude`, and `Longitude` are excluded from the initial churn model because they mainly identify location rather than customer behavior and can introduce unnecessary high-cardinality signals. They remain available for geographic business analysis.


In [12]:
# Review special-purpose columns
special_columns = [
    "CustomerID", "Count", "Country", "State",
    "City", "Zip Code", "Lat Long", "Latitude", "Longitude",
    "Churn Label", "Churn Value", "Churn Score", "Churn Reason"
]

pd.DataFrame({
    "column": special_columns,
    "unique_values": [df_raw[col].nunique(dropna=False) for col in special_columns],
    "missing_count": [df_raw[col].isna().sum() for col in special_columns],
})


,column,unique_values,missing_count
0,CustomerID,7043,0
1,Count,1,0
2,Country,1,0
3,State,1,0
4,City,1129,0
5,Zip Code,1652,0
6,Lat Long,1652,0
7,Latitude,1652,0
8,Longitude,1651,0
9,Churn Label,2,0


## 5. Clean the data

In [13]:
# Work on a copy so the raw data remains unchanged
df = df_raw.copy()

# Standardize text values without changing numeric values stored in mixed object columns
object_cols = df.select_dtypes(include="object").columns
for col in object_cols:
    df[col] = df[col].map(lambda x: x.strip() if isinstance(x, str) else x)

# Total Charges is logically numeric; convert defensively
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")

# Remove exact duplicate rows if present
df = df.drop_duplicates().reset_index(drop=True)

# Validate target consistency
expected_churn_value = df["Churn Label"].map({"No": 0, "Yes": 1})
inconsistent_target_rows = (expected_churn_value != df["Churn Value"]).sum()

print(f"Rows after duplicate removal: {len(df):,}")
print(f"Inconsistent Churn Label / Churn Value rows: {inconsistent_target_rows:,}")
print(f"Missing Total Charges after numeric conversion: {df['Total Charges'].isna().sum():,}")


Rows after duplicate removal: 7,043
Inconsistent Churn Label / Churn Value rows: 0
Missing Total Charges after numeric conversion: 11


In [14]:
# Investigate any Total Charges values that became missing after conversion
df.loc[df["Total Charges"].isna(), [
    "CustomerID", "Tenure Months", "Monthly Charges", "Total Charges",
    "Churn Label"
]]


,CustomerID,Tenure Months,Monthly Charges,Total Charges,Churn Label
2234,4472-LVYGI,0,52.55,NaN,No
2438,3115-CZMZD,0,20.25,NaN,No
2568,5709-LVOEQ,0,80.85,NaN,No
2667,4367-NUYAO,0,25.75,NaN,No
2856,1371-DWPAZ,0,56.05,NaN,No
4331,7644-OMVMY,0,19.85,NaN,No
4687,3213-VVOLG,0,25.35,NaN,No
5104,2520-SGTTA,0,20.00,NaN,No
5719,2923-ARZLG,0,19.70,NaN,No
6772,4075-WKNIU,0,73.35,NaN,No


If `Total Charges` is missing for customers with zero tenure, the missing value reflects customers who have not accumulated charges yet rather than an unknown historical amount. For those records, `Total Charges` is set to `0`.

Any remaining missing values are left as missing and reported for review rather than imputed.

In [15]:
# Fill structurally missing Total Charges for brand-new (0-month tenure) customers
zero_tenure_missing = df["Total Charges"].isna() & df["Tenure Months"].eq(0)
df.loc[zero_tenure_missing, "Total Charges"] = 0

remaining_total_charge_missing = df["Total Charges"].isna().sum()

print(f"Zero-tenure Total Charges filled with 0: {zero_tenure_missing.sum():,}")
print(f"Remaining missing Total Charges: {remaining_total_charge_missing:,}")


Zero-tenure Total Charges filled with 0: 11
Remaining missing Total Charges: 0


## 6. Basic target profile

In [16]:
# Churn distribution
churn_profile = (
    df["Churn Value"]
    .value_counts()
    .rename(index={0: "Retained", 1: "Churned"})
    .to_frame("customers")
)

churn_profile["pct"] = 100 * churn_profile["customers"] / len(df)
churn_profile


,customers,pct
Churn Value,,
Retained,5174,73.46
Churned,1869,26.54


## 7. Create a leakage-safe modeling dataset


In [17]:
TARGET = "Churn Value"

# Columns intentionally excluded from predictive modeling
MODEL_EXCLUDE = [
    # Identifier / technical helper
    "CustomerID",
    "Count",

    # Constant columns
    "Country",
    "State",

    # Exact / high-cardinality location fields
    "City",
    "Zip Code",
    "Lat Long",
    "Latitude",
    "Longitude",

    # Target duplicate / post-outcome leakage
    "Churn Label",
    "Churn Score",
    "Churn Reason",
]

modeling_df = df.drop(columns=MODEL_EXCLUDE).copy()

print(f"Business-analysis dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Modeling dataset:          {modeling_df.shape[0]:,} rows × {modeling_df.shape[1]} columns")
print(f"Target column:             {TARGET}")


Business-analysis dataset: 7,043 rows × 33 columns
Modeling dataset:          7,043 rows × 21 columns
Target column:             Churn Value


In [18]:
# Confirm target and candidate model features
feature_columns = [col for col in modeling_df.columns if col != TARGET]

print(f"Number of candidate features: {len(feature_columns)}")
print("\nCandidate features:")
for col in feature_columns:
    print(f"- {col}")


Number of candidate features: 20

Candidate features:
- Gender
- Senior Citizen
- Partner
- Dependents
- Tenure Months
- Phone Service
- Multiple Lines
- Internet Service
- Online Security
- Online Backup
- Device Protection
- Tech Support
- Streaming TV
- Streaming Movies
- Contract
- Paperless Billing
- Payment Method
- Monthly Charges
- Total Charges
- CLTV


In [19]:
# Final modeling data quality check
final_quality = pd.DataFrame({
    "dtype": modeling_df.dtypes.astype(str),
    "missing_count": modeling_df.isna().sum(),
    "unique_values": modeling_df.nunique(dropna=False),
}).sort_index()

final_quality


,dtype,missing_count,unique_values
CLTV,int64,0,3438
Churn Value,int64,0,2
Contract,object,0,3
Dependents,object,0,2
Device Protection,object,0,3
Gender,object,0,2
Internet Service,object,0,3
Monthly Charges,float64,0,1585
Multiple Lines,object,0,3
Online Backup,object,0,3


## 8. Data dictionary

In [20]:
column_roles = {
    "CustomerID": ("Identifier", "Business analysis only"),
    "Count": ("Technical helper", "Exclude"),
    "Country": ("Geography", "Exclude from model — constant"),
    "State": ("Geography", "Exclude from model — constant"),
    "City": ("Geography", "Business/geographic analysis only"),
    "Zip Code": ("Geography", "Business/geographic analysis only"),
    "Lat Long": ("Geography", "Exclude — redundant"),
    "Latitude": ("Geography", "Business/geographic analysis only"),
    "Longitude": ("Geography", "Business/geographic analysis only"),
    "Gender": ("Demographic", "Model candidate"),
    "Senior Citizen": ("Demographic", "Model candidate"),
    "Partner": ("Demographic", "Model candidate"),
    "Dependents": ("Demographic", "Model candidate"),
    "Tenure Months": ("Account", "Model candidate"),
    "Phone Service": ("Service", "Model candidate"),
    "Multiple Lines": ("Service", "Model candidate"),
    "Internet Service": ("Service", "Model candidate"),
    "Online Security": ("Service", "Model candidate"),
    "Online Backup": ("Service", "Model candidate"),
    "Device Protection": ("Service", "Model candidate"),
    "Tech Support": ("Service", "Model candidate"),
    "Streaming TV": ("Service", "Model candidate"),
    "Streaming Movies": ("Service", "Model candidate"),
    "Contract": ("Account", "Model candidate"),
    "Paperless Billing": ("Billing", "Model candidate"),
    "Payment Method": ("Billing", "Model candidate"),
    "Monthly Charges": ("Billing", "Model candidate"),
    "Total Charges": ("Billing", "Model candidate"),
    "Churn Label": ("Outcome", "Exclude — target duplicate"),
    "Churn Value": ("Outcome", "Prediction target"),
    "Churn Score": ("Outcome-derived", "Exclude — leakage"),
    "CLTV": ("Customer value", "Model candidate / retention prioritization"),
    "Churn Reason": ("Post-churn outcome", "Business analysis only — leakage"),
}

data_dictionary = pd.DataFrame([
    {
        "column": col,
        "dtype": str(df[col].dtype),
        "role": column_roles[col][0],
        "project_use": column_roles[col][1],
    }
    for col in df.columns
])

data_dictionary


,column,dtype,role,project_use
0,CustomerID,object,Identifier,Business analysis only
1,Count,int64,Technical helper,Exclude
2,Country,object,Geography,Exclude from model — constant
3,State,object,Geography,Exclude from model — constant
4,City,object,Geography,Business/geographic analysis only
5,Zip Code,int64,Geography,Business/geographic analysis only
6,Lat Long,object,Geography,Exclude — redundant
7,Latitude,float64,Geography,Business/geographic analysis only
8,Longitude,float64,Geography,Business/geographic analysis only
9,Gender,object,Demographic,Model candidate


## 9. Export prepared datasets

Two files are produced:

1. **`telco_churn_cleaned.csv`** — full cleaned customer dataset for SQL, EDA, churn-driver analysis, geographic analysis, and retention analysis.
2. **`telco_churn_modeling.csv`** — leakage-safe dataset for the later Python machine-learning notebook.

The files are saved to `data/processed/`.


In [21]:
# Choose an output directory
repo_processed_dir = Path("../data/processed")
local_processed_dir = Path("data/processed")

# Prefer repository-style output when notebook is inside notebooks/
if Path("../data").exists():
    OUTPUT_DIR = repo_processed_dir
else:
    OUTPUT_DIR = local_processed_dir

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cleaned_path = OUTPUT_DIR / "telco_churn_cleaned.csv"
modeling_path = OUTPUT_DIR / "telco_churn_modeling.csv"
dictionary_path = OUTPUT_DIR / "telco_churn_data_dictionary.csv"

df.to_csv(cleaned_path, index=False)
modeling_df.to_csv(modeling_path, index=False)
data_dictionary.to_csv(dictionary_path, index=False)

print(f"Saved: {cleaned_path}")
print(f"Saved: {modeling_path}")
print(f"Saved: {dictionary_path}")


Saved: ../data/processed/telco_churn_cleaned.csv
Saved: ../data/processed/telco_churn_modeling.csv
Saved: ../data/processed/telco_churn_data_dictionary.csv
